# 04: 虚拟双敲除 + 动态 ODE
## In-silico 双敲除 → 主动学习闭环

验证完整闭环:
1. MB-PLS 潜变量 → GEM 约束
2. 批量虚拟双敲除 (FBA)
3. 主动学习选最优双敲除
4. 糖酵解 ODE 动态模拟

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from chemocalib.models.mbpls import MultiBlockPLS, generate_toy_multiblock_data
from chemocalib.gem.fba import FBASimulator
from chemocalib.gem.constraints import LatentToConstraint
from chemocalib.virtual_experiment.knockout import DoubleKnockoutDesigner
from chemocalib.active_learning.uncertainty import UncertaintySampler
from chemocalib.dynamic_layer.ode_solver import GlycolysisODE

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# 加载 GEM 模型
sim = FBASimulator(model_name='textbook')
sim.load_model()
print(sim.summary())

# 野生型 FBA
wt = sim.wild_type_fba()
print(f'野生型生物量: {wt["objective_value"]:.4f}')

In [ ]:
# MB-PLS 训练 + 约束生成
blocks, y, feature_names = generate_toy_multiblock_data(n_samples=60, seed=42)
mbpls = MultiBlockPLS(n_components=5, block_names=['代谢组', '转录组', '蛋白组'])
mbpls.fit(blocks, y)

gem_metabolites = sim.get_exchange_reactions()
gem_met_ids = [m.replace('EX_', '') for m in gem_metabolites]

mapper = LatentToConstraint(scaling_mode='soft')
mapper.build_feature_reaction_map(
    feature_names[0],
    gem_met_ids[:len(feature_names[0])],
    vip_scores=mbpls.vip_scores[0],
)

latent_mean = mbpls.super_scores.mean(axis=0)
bounds = mapper.latent_to_bounds(latent_mean, n_components=3)

fba_res = sim.fba_with_chemometric_constraints(bounds)
print(f'约束前 生物量: {wt["objective_value"]:.4f}')
print(f'约束后 生物量: {fba_res["objective_value"]:.4f}')
print(f'施加约束: {fba_res["constraints_applied"]}/{fba_res["constraints_total"]}')

In [ ]:
# 虚拟双敲除 (100 对, 轻薄本 ~10s)
genes = sim.get_all_genes()
designer = DoubleKnockoutDesigner(gene_pool=genes, design_strategy='exhaustive')
designer.generate_pairs(n_pairs=100)
dko = designer.run_virtual_experiments(sim, verbose=True)

# 生长率分布
growth = dko['growth_double'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(growth, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(wt['objective_value'], color='red', linestyle='--', label=f'WT={wt["objective_value"]:.2f}')
axes[0].set_xlabel('Growth Rate')
axes[0].set_ylabel('Count')
axes[0].set_title('Double KO Growth Distribution')
axes[0].legend()

axes[1].scatter(dko['gene_a'].apply(hash) % 100, dko['gene_b'].apply(hash) % 100,
               c=dko['growth_ratio'], cmap='RdYlGn', s=50, edgecolor='k', alpha=0.7)
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Growth Ratio')
axes[1].set_xlabel('Gene A (hashed)')
axes[1].set_ylabel('Gene B (hashed)')
axes[1].set_title('Double KO Phenotype Landscape')
plt.tight_layout()
plt.show()

In [ ]:
# 主动学习: 选最优双敲除
residuals = mbpls.residual_space(blocks)
sampler = UncertaintySampler(strategy='hybrid')

gene_pairs = list(zip(dko['gene_a'], dko['gene_b']))
candidates = sampler.select_double_knockout_candidates(
    all_gene_pairs=gene_pairs,
    pair_features=dko[['growth_ratio']].values,
    residuals=residuals,
    n_select=10,
    n_pool=min(200, len(gene_pairs)),
)

print('\n【给合作者真做的 Top 10 双敲除】\n')
for _, row in candidates.iterrows():
    print(f"  #{int(row['rank']):2d}  敲除 {row['gene_a']:10s} + {row['gene_b']:10s}  (不确定性: {row['uncertainty']:.4f})")

In [ ]:
# 糖酵解 ODE 动态模拟
ode = GlycolysisODE()
ode.calibrate_from_latent(latent_mean, n_component=0)
result = ode.simulate(t_span=(0, 60), n_points=300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 浓度变化
ax = axes[0]
ax.plot(result['t'], result['G6P'], label='G6P', linewidth=2)
ax.plot(result['t'], result['FBP'], label='FBP', linewidth=2)
ax.plot(result['t'], result['PYR'], label='PYR', linewidth=2)
ax.set_xlabel('Time')
ax.set_ylabel('Concentration')
ax.set_title('Glycolysis ODE Dynamics')
ax.legend()

# 通量
ax = axes[1]
ax.plot(result['t'], result['fluxes']['v_HK'], label='v_HK', linewidth=2)
ax.plot(result['t'], result['fluxes']['v_PFK'], label='v_PFK', linewidth=2)
ax.plot(result['t'], result['fluxes']['v_PK'], label='v_PK', linewidth=2)
ax.set_xlabel('Time')
ax.set_ylabel('Flux')
ax.set_title('Reaction Fluxes')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Kcat proxies: {ode.extract_kcat_proxies()}')

In [ ]:
print('\n' + '=' * 60)
print(' 闭环验证完成 ✓')
print(' 感知(MB-PLS) → 预测(GEM约束+FBA) → 选样(主动学习)')
print('=' * 60)